# 🌸 Notebook 1: What Is a Bloom Filter?

**The big question:** *"Have I seen this URL before?"*

Imagine you are building a web crawler like Googlebot. Every second it finds thousands of new links. Before crawling a page you want to ask: **have I already crawled this URL?** If yes, skip it. If no, add it to the queue.

Simple question — but doing it **fast** and **cheaply** when you have billions of URLs is surprisingly hard.

In this notebook we compare three approaches:

1. 🟥 **BAD** — store every URL in a Python `list` and scan it.
2. 🟨 **BETTER** — use a Python `set` (hash table).
3. 🟩 **BEST** — build a **bloom filter** ourselves from raw bits and hash functions.

## Learning objectives
- Understand why `list` and `set` are both wrong at scale (time vs memory).
- Learn the bloom filter algorithm by implementing it from scratch.
- See false positives happen in practice, and prove false negatives *cannot* happen.

## 🛠️ Setup

This lab has **zero Docker dependencies** — it is pure Python.

```bash
cd 01-foundations/bloom-filters
uv sync
```

### Kernel selection

Select the `.venv` kernel in the VS Code kernel picker (top-right of this notebook).

If the kernel does not appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 🟥 Approach 1 (BAD): a Python list

The simplest thing that could possibly work: keep every URL in a list. To check membership, scan the list.

- **Lookup time:** O(n). If you have 10 million URLs, every check reads up to 10 million strings.
- **Memory:** stores the full string of every URL.

Let's measure it.

In [1]:
import time
import random
import string

def random_url() -> str:
    slug = ''.join(random.choices(string.ascii_lowercase, k=12))
    return f"https://example.com/{slug}"

random.seed(42)
urls = [random_url() for _ in range(100_000)]

# Put them in a list
seen_list = list(urls)

# Pick a URL that IS in the list and one that is NOT
hit = urls[-1]
miss = "https://example.com/definitely-not-there"

def time_lookup(fn, n=1000):
    start = time.perf_counter()
    for _ in range(n):
        fn()
    return (time.perf_counter() - start) * 1000 / n  # ms per lookup

print(f"list hit  : {time_lookup(lambda: hit in seen_list):.4f} ms")
print(f"list miss : {time_lookup(lambda: miss in seen_list):.4f} ms")

list hit  : 0.5107 ms


list miss : 0.3764 ms


The **miss** case is the worst: Python has to scan the entire list before concluding the URL is not there. This is exactly the case a crawler hits most often (new URLs).

## 🟨 Approach 2 (BETTER): a Python set

A `set` is a hash table. Lookups are **O(1) average**. Much faster!

But there is a catch: a `set` stores the actual URL strings in memory. If each URL is ~60 bytes and you have 1 billion of them, that's ~60 GB of RAM just for the membership check. Most servers cannot afford that.

In [2]:
seen_set = set(urls)

print(f"set hit   : {time_lookup(lambda: hit in seen_set):.6f} ms")
print(f"set miss  : {time_lookup(lambda: miss in seen_set):.6f} ms")

import sys
print(f"\nMemory used by the set of 100k URLs: {sys.getsizeof(seen_set):,} bytes (the set object alone)")
print("Plus each string inside, roughly 60 bytes × 100_000 ≈ 6 MB more.")

set hit   : 0.000030 ms
set miss  : 0.000028 ms

Memory used by the set of 100k URLs: 4,194,520 bytes (the set object alone)
Plus each string inside, roughly 60 bytes × 100_000 ≈ 6 MB more.


Speed: great. Memory: grows **linearly** with the number of URLs. For a web-scale crawler this is the bottleneck.

**Can we do O(1) lookups without storing the strings at all?** Yes — that is exactly what a bloom filter does.

## 🟩 Approach 3 (BEST): a bloom filter from scratch

A bloom filter is just two things:

1. A **bit array** of size `m` — a long row of 0s and 1s, initially all 0.
2. **k hash functions** that each map an item to a position in that bit array.

### Adding an item
Run the item through all `k` hash functions. Each gives you a position. Set those bit positions to 1.

### Checking an item
Run the item through the same `k` hash functions. If **any** of those bits is 0 → the item is **definitely not** in the set. If **all** of those bits are 1 → the item is **probably** in the set.

### Why probably?
Because other items may have set the same bits by accident (a **false positive**). But the filter can **never give a false negative**: if you added it, all its bits are 1, period.

### Analogy: stamps on an envelope
Imagine a row of numbered boxes, all empty. When you see a URL, you stamp it with 3 special stampers — each stamper always stamps the *same* 3 boxes for the *same* URL. Later you can ask "did I see this URL?" by re-running the 3 stampers. If even one of those boxes is empty, you know for sure you never saw it. If all 3 are stamped, you *probably* saw it — but maybe 3 other URLs together stamped those boxes first.

### Double hashing: how to get `k` hashes from 2

We do not really want `k` different hash functions. A classic trick (Kirsch & Mitzenmacher, 2006) is:

```
h_i(x) = (h1(x) + i * h2(x)) mod m    for i = 0, 1, ..., k-1
```

Using `hashlib.sha256` once, we split the digest into two 64-bit integers `h1` and `h2` and generate all `k` positions from them. Fast and good enough.

In [3]:
import hashlib
from pydantic import BaseModel, Field

class BloomFilter:
    """A tiny educational bloom filter built on a plain bytearray.

    m = number of bits, k = number of hash functions.
    We store the bits packed into a bytearray of length ceil(m/8).
    """

    def __init__(self, m: int, k: int):
        if m <= 0 or k <= 0:
            raise ValueError("m and k must be positive")
        self.m = m
        self.k = k
        # ceil(m/8) bytes, all zero bits to start
        self.bits = bytearray((m + 7) // 8)

    # ---- bit-level helpers ----
    def _set_bit(self, i: int) -> None:
        self.bits[i // 8] |= 1 << (i % 8)

    def _get_bit(self, i: int) -> int:
        return (self.bits[i // 8] >> (i % 8)) & 1

    # ---- the double-hashing trick ----
    def _positions(self, item: str):
        data = item.encode("utf-8")
        digest = hashlib.sha256(data).digest()  # 32 bytes
        h1 = int.from_bytes(digest[:8], "big")
        h2 = int.from_bytes(digest[8:16], "big")
        for i in range(self.k):
            yield (h1 + i * h2) % self.m

    # ---- public API ----
    def add(self, item: str) -> None:
        for pos in self._positions(item):
            self._set_bit(pos)

    def __contains__(self, item: str) -> bool:
        return all(self._get_bit(pos) for pos in self._positions(item))

    def memory_bytes(self) -> int:
        return len(self.bits)

print("BloomFilter class defined ✅")

BloomFilter class defined ✅


### A tiny sanity check

Let's add a handful of URLs, then verify:
- Everything we added is reported as present (**no false negatives, ever**).
- Things we didn't add are *mostly* reported as absent (but sometimes not — false positives).

In [4]:
bf = BloomFilter(m=64, k=3)  # deliberately tiny to see false positives
added = ["https://example.com/a", "https://example.com/b", "https://example.com/c"]
for u in added:
    bf.add(u)

print("Items we added (should all be True):")
for u in added:
    print(f"  {u!r:40s} -> {u in bf}")

print("\nItems we did NOT add (mostly False, maybe a False if unlucky):")
probes = [f"https://example.com/other-{i}" for i in range(10)]
for u in probes:
    print(f"  {u!r:40s} -> {u in bf}")

Items we added (should all be True):
  'https://example.com/a'                  -> True
  'https://example.com/b'                  -> True
  'https://example.com/c'                  -> True

Items we did NOT add (mostly False, maybe a False if unlucky):
  'https://example.com/other-0'            -> False
  'https://example.com/other-1'            -> False
  'https://example.com/other-2'            -> False
  'https://example.com/other-3'            -> False
  'https://example.com/other-4'            -> False
  'https://example.com/other-5'            -> False
  'https://example.com/other-6'            -> False
  'https://example.com/other-7'            -> False
  'https://example.com/other-8'            -> False
  'https://example.com/other-9'            -> False


## 🧪 Experiment: false positives happen, false negatives don't

Let's insert 1,000 URLs into a filter with only 2,000 bits (on purpose — this will cause lots of collisions) and then probe 10,000 fresh URLs to see the false-positive rate. We'll also verify every inserted URL is still reported as present.

In [5]:
bf = BloomFilter(m=2000, k=4)

inserted = [random_url() for _ in range(1000)]
for u in inserted:
    bf.add(u)

# False negatives?
false_negatives = sum(1 for u in inserted if u not in bf)
print(f"False negatives: {false_negatives}  (must be 0)")

# False positives?
inserted_set = set(inserted)
probes = [random_url() for _ in range(10_000)]
probes = [p for p in probes if p not in inserted_set]  # only true negatives
false_positives = sum(1 for p in probes if p in bf)
print(f"False positives: {false_positives} / {len(probes)}  "
      f"({100*false_positives/len(probes):.2f}%)")

False negatives: 0  (must be 0)
False positives: 5820 / 10000  (58.20%)


### What did we gain?

| Approach | Time per lookup | Memory for N items |
|---|---|---|
| list    | O(N) | stores full items |
| set     | O(1) | stores full items |
| **bloom** | **O(k) ≈ O(1)** | **just a bit array — independent of item size!** |

The filter above used **2000 bits = 250 bytes** to track 1000 URLs. A `set` would store all 1000 full URL strings (tens of KB). That's the bloom filter superpower: **membership without memory of the items themselves**.

### The catch

- ❌ You **cannot delete** items (unsetting a bit could break other items that share it).
- ❌ You **cannot list** the items.
- ❌ You get **false positives**. Tune `m` and `k` to keep them rare.
- ✅ You **never** get false negatives.

👉 Next notebook: how to *choose* `m` and `k` to hit a target false-positive rate.